<div style="background:linear-gradient(135deg,#1A5276 0%,#2E86C1 100%);padding:40px 36px 32px 36px;border-radius:10px;margin-bottom:8px;">
  <p style="color:#AED6F1;font-size:13px;margin:0 0 6px 0;letter-spacing:2px;">CURSO 8 · MÓDULO 2 · CLASE 7</p>
  <h1 style="color:white;font-size:36px;margin:0 0 10px 0;font-weight:700;">Aplicación: Scoring de Crédito y Threshold Óptimo</h1>
  <p style="color:#AED6F1;font-size:16px;margin:0 0 24px 0;font-style:italic;">Logística en producción — caso completo de industria</p>
  <hr style="border-color:#5DADE2;margin:0 0 20px 0;">
  <p style="color:#EBF5FB;font-size:13px;margin:0;">📌 <strong>Docente:</strong> Josef Rodriguez &nbsp;·&nbsp; <strong>Duración:</strong> 2 horas</p>
</div>

## Objetivos
| # | Al terminar podés |
|---|-------------------|
| 1 | Elegir el threshold óptimo según el costo del negocio |
| 2 | Construir una matriz de confusión ponderada por costos |
| 3 | Comparar logística vs LDA en el mismo dataset |
| 4 | Entregar un reporte ejecutivo de scoring |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.special import expit
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
np.set_printoptions(precision=4, suppress=True)
plt.rcParams.update({'figure.dpi':110,'font.size':11,'axes.spines.top':False,'axes.spines.right':False})
SEED=42; np.random.seed(SEED)
print('✅ Setup listo')

---
## 1. El problema del threshold en negocio

El threshold 0.5 es raramente el óptimo en negocio. El trade-off depende de los **costos**:
- **FN (falso negativo):** no detectar un default → pérdida del capital prestado
- **FP (falso positivo):** rechazar a un buen cliente → pérdida de ingreso por intereses

In [ ]:
np.random.seed(SEED)
n_bank=3000
ingresos=np.random.normal(45000,18000,n_bank)
antigu=np.random.randint(0,25,n_bank).astype(float)
deuda=np.random.uniform(0.05,0.85,n_bank)
prod=np.random.randint(1,5,n_bank).astype(float)
logit_b=-2-0.00003*ingresos+0.01*antigu+3.2*deuda-0.2*prod
y_bank=np.random.binomial(1,expit(logit_b))

feats=['ingresos','antiguedad','deuda_ratio','productos']
X_raw=np.column_stack([ingresos,antigu,deuda,prod])
sc=StandardScaler(); X_sc=sc.fit_transform(X_raw)
Xtr,Xte,ytr,yte=train_test_split(X_sc,y_bank,test_size=0.25,random_state=SEED)

lr=LogisticRegression(C=1e6,solver='lbfgs',max_iter=500)
lr.fit(Xtr,ytr); proba=lr.predict_proba(Xte)[:,1]
print(f'Tasa de default: {y_bank.mean():.1%}')
print(f'AUC: {roc_auc_score(yte,proba):.4f}')

In [ ]:
# Análisis costo-beneficio para distintos thresholds
# Suposición: préstamo promedio = 10,000 USD
# FN (perder préstamo) = -10,000 USD
# FP (perder cliente) = -500 USD (margen de interés perdido)
COSTO_FN = 10_000
COSTO_FP = 500

from sklearn.metrics import confusion_matrix

thresholds = np.arange(0.1, 0.9, 0.05)
costos = []

for th in thresholds:
    y_pred = (proba >= th).astype(int)
    tn,fp,fn,tp = confusion_matrix(yte,y_pred).ravel()
    costo_total = fn*COSTO_FN + fp*COSTO_FP
    costos.append(costo_total)

opt_th = thresholds[np.argmin(costos)]
fig,ax=plt.subplots(figsize=(9,4))
ax.plot(thresholds,costos,'o-',color='#E74C3C',lw=2)
ax.axvline(opt_th,color='#27AE60',lw=2,linestyle='--',label=f'Óptimo={opt_th:.2f}')
ax.set(xlabel='Threshold',ylabel='Costo total (USD)',title='Costo de negocio por threshold')
ax.legend(); ax.grid(True,alpha=0.25); plt.tight_layout(); plt.show()
print(f'Threshold óptimo: {opt_th:.2f}  Costo mínimo: {min(costos):,.0f} USD')
y_opt=(proba>=opt_th).astype(int)
print(classification_report(yte,y_opt,target_names=['No default','Default']))

---
## 2. Logística vs LDA — ¿cuándo usar cada uno?

| Aspecto | Logística | LDA |
|---------|-----------|-----|
| Supuesto | Ninguno sobre X | X ~ MVN por clase |
| Estimación | MLE (log-lik) | Máx. varianza entre clases |
| Output | P(y=1\|x) directamente | Scores + probabilidades |
| Robusto | Más (no asume normalidad) | Menos, pero más eficiente si se cumple |

In [ ]:
# Comparación directa en el mismo dataset
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

lda=LinearDiscriminantAnalysis(); lda.fit(Xtr,ytr)
lr2=LogisticRegression(C=1e6,solver='lbfgs',max_iter=500); lr2.fit(Xtr,ytr)

modelos={'Logística':lr2,'LDA':lda}
print('Comparación de modelos:')
print(f'{"Modelo":15s} {"Accuracy":>10s} {"AUC":>10s}')
print('─'*38)
for nm,m in modelos.items():
    acc=accuracy_score(yte,m.predict(Xte))
    auc=roc_auc_score(yte,m.predict_proba(Xte)[:,1])
    print(f'  {nm:13s} {acc:>10.4f} {auc:>10.4f}')
print('\n→ Cuando los datos son aproximadamente normales, LDA y Logística dan resultados similares.')
print('→ Con outliers o distribuciones asimétricas, Logística es más robusta.')

---
## 3. Reporte ejecutivo de scoring

In [ ]:
# Simular nuevos solicitantes
nuevos_raw=np.array([[70000,8,0.22,3],[18000,1,0.82,1],[45000,12,0.45,2]])
nuevos_sc=sc.transform(nuevos_raw)
p_nuevos=lr.predict_proba(nuevos_sc)[:,1]
y_nuevos=(p_nuevos>=opt_th).astype(int)

print('┌─────────────────────────────────────────────────┐')
print('│         REPORTE EJECUTIVO DE SCORING             │')
print('├─────────────────────────────────────────────────┤')
print(f'│  Modelo: Regresión Logística │ AUC={roc_auc_score(yte,proba):.3f}      │')
print(f'│  Threshold óptimo: {opt_th:.2f}  │  n_train={len(ytr):,}        │')
print('├─────────────────────────────────────────────────┤')
print('│  Nuevos solicitantes:                            │')
for i,(p,dec) in enumerate(zip(p_nuevos,y_nuevos)):
    status='APROBAR ✓' if dec==0 else 'RECHAZAR ✗'
    print(f'│  Cliente {i+1}: P(default)={p:.3f}  →  {status:12s}  │')
print('└─────────────────────────────────────────────────┘')

---
## Conclusiones
<div style="background:#EBF5FB;border-left:5px solid #2E86C1;padding:20px 24px;border-radius:0 8px 8px 0;">

**01 · El threshold no es 0.5 — es una decisión de negocio**
El threshold óptimo depende del costo relativo de FN vs FP. En crédito, FN es mucho más caro.

**02 · Logística y LDA son complementarios**
LDA es más eficiente cuando los supuestos se cumplen. Logística es más robusta cuando no.

**03 · AUC separa el modelo del threshold**
AUC mide la calidad del modelo independiente del threshold. Solo después se elige el threshold óptimo.

</div>

<div style="background:#1A5276;color:white;padding:20px 24px;border-radius:8px;"><strong>Próxima clase</strong><br>Módulo 3 — Análisis Discriminante Lineal (LDA) · Funciones discriminantes · Fisher<br><em>Docente: Josef Rodriguez · Curso 8</em></div>